
Assignment 13

MLflow | Deployment | Monitoring & Maintenance | Pipelines & Automation

## Question 1: What is MLflow, and what are its major components? Explain the roles of MLflow Tracking, Projects, Models, and Model Registry.

**Answer:**

MLflow is an open-source platform (created by Databricks) for managing the complete machine-learning lifecycle — experimentation, reproducibility, deployment and a central model store. It is library-agnostic (works with scikit-learn, TensorFlow, PyTorch, XGBoost, Spark MLlib and others), language-agnostic (Python, R, Java, REST API) and can run locally, on a server or in the cloud.

It solves the practical problems of ML development: experiments are hard to track, results are hard to reproduce, there is no standard way to package a model, and there is no central place to manage model versions.

### 1. MLflow Tracking
- Role: Records and queries experiments — it is the logging and comparison layer.
- For each run it logs: parameters (hyperparameters such as learning rate, n_estimators), metrics (accuracy, RMSE, F1 — can be logged over time/epochs), artifacts (model files, plots, confusion matrices, data samples), tags and notes, the source code version (Git commit), and the start/end time and author.
- Provides a web UI to compare runs side by side, sort and filter by metric, and visualise parameter–metric relationships.
- Supports autologging (mlflow.sklearn.autolog(), mlflow.tensorflow.autolog()) which captures everything automatically with one line.
- Runs are grouped into experiments, and runs can be nested (useful for hyperparameter sweeps).

In [ ]:
import mlflow
mlflow.set_experiment("churn-model")
with mlflow.start_run():
    mlflow.log_param("n_estimators", 200)
    mlflow.log_metric("f1", 0.87)
    mlflow.sklearn.log_model(model, "model")


### 2. MLflow Projects
- Role: Packages ML code in a reusable and reproducible format so anyone can rerun it identically.
- A project is simply a directory (or Git repository) containing an MLproject file that declares the entry points, their parameters and the environment (Conda, virtualenv, Docker or a system environment).
- It can be executed with a single command, locally or remotely: mlflow run . -P alpha=0.5 or mlflow run https://github.com/user/repo.
- Why it matters: it removes "it works on my machine" problems, enables running the same training job on another machine, a cluster (Databricks, Kubernetes) or in CI, and makes multi-step pipelines chainable.

### 3. MLflow Models
- Role: Defines a standard packaging format for a trained model so that one model can be deployed to many different serving platforms.
- A saved model directory contains an MLmodel file describing one or more "flavours" — for example the native flavour (sklearn, pytorch) plus the generic python_function (pyfunc) flavour that any tool can load and call.
- It also stores the dependencies (conda.yaml / requirements.txt), an optional model signature (expected input and output schema) and input examples, which enables validation at serving time.
- Deployment targets: a local REST server (mlflow models serve), Docker container, AWS SageMaker, Azure ML, Databricks, Kubernetes, or batch scoring with Spark UDF.
- Why it matters: "package once, deploy anywhere" — the deployment code does not need to know which library trained the model.

### 4. MLflow Model Registry
- Role: A centralised model store providing versioning, lifecycle management, annotation and governance for models.
- A registered model has a name; each time a model is registered under that name it gets an incrementing version number (Version 1, 2, 3 …).
- Each version moves through stages: None → Staging → Production → Archived, with stage transitions that can require approval.
- Stores lineage (which run, code commit and data produced this version), descriptions, tags and the full history of who promoted what and when.
- Serving code can reference a model by stage — models:/churn_model/Production — so the deployment automatically picks up whichever version is currently promoted.
- Why it matters: it enables safe promotion, instant rollback to a previous version, A/B or champion–challenger testing, collaboration across teams and a complete audit trail for compliance.

Newer MLflow versions add further components such as MLflow Recipes (opinionated pipeline templates), MLflow Evaluate and LLM/GenAI tracking with prompt engineering support.

## Question 2: What problem does MLflow Tracking solve? Why is tracking experiments important when developing ML models?

**Answer:**

The problem it solves

Model development is inherently iterative and experimental. A data scientist may run hundreds of experiments varying algorithms, hyperparameters, feature sets and preprocessing steps. Without a tracking system, this information is scattered and quickly lost:
- Results are kept in notebook cells, Excel sheets, text files, screenshots or the scientist's memory — all of which get overwritten or lost.
- Model files are named model_final.pkl, model_final_v2.pkl, model_final_REAL.pkl with no record of what is inside them.
- Nobody can answer "which combination of parameters produced the 0.91 F1 score last month?"
- The best model cannot be reproduced because the code, data version and environment that produced it are unknown.
- Comparing 50 runs manually is impractical and error-prone.
- Team members repeat experiments that a colleague has already tried and rejected.
- There is no audit trail for regulated environments.

MLflow Tracking solves this by providing one automatic, centralised, queryable record of every run — parameters, metrics, artifacts, code version, environment and timestamp — plus a UI to compare them.

Why experiment tracking is important
- Reproducibility: every result can be traced back to the exact code commit, parameters, data version and environment that produced it, so it can be recreated on demand.
- Objective model selection: dozens of runs can be sorted and filtered by any metric, so the genuinely best model is chosen instead of the most recently remembered one.
- Faster iteration: the team can see what has already been tried and what direction is improving results, avoiding duplicated work and dead ends.
- Collaboration: results are visible to the whole team from a shared server, not trapped in one person's notebook.
- Debugging and root-cause analysis: when production performance drops, the run history shows exactly what changed between model versions.
- Governance, auditability and compliance: regulated industries (finance, healthcare) must prove which data and which code produced a deployed model — the tracking log provides that evidence.
- Smooth handoff to deployment: the logged model artifact, signature and dependencies feed directly into the Model Registry and CI/CD, so there is no manual repackaging step.
- Hyperparameter tuning insight: logged runs can be visualised to understand which parameters actually matter.

## Question 3: What is the difference between batch inference and real-time inference? Discuss the advantages, disadvantages, and suitable use cases for both.

**Answer:**

Batch inference (offline inference)

Predictions are generated for a large group of records at once, on a schedule (hourly, nightly, weekly), and the results are stored in a database, data warehouse or file store. The application later reads the pre-computed prediction instead of calling the model.

Advantages:
- Very high throughput — millions of records processed efficiently using vectorised/distributed compute (Spark, Databricks, BigQuery).
- Cost-efficient: compute is used only during the batch window and can use cheap spot/preemptible instances; no always-on servers.
- Simpler infrastructure: no low-latency API, load balancer or autoscaling needed; a scheduled job is enough.
- Complex, expensive features are affordable because there is no latency budget — large aggregations, joins across many tables, heavy models.
- Easier to test, monitor and debug: the output is a dataset that can be validated before being published; a failed batch can simply be rerun.

Disadvantages:
- Predictions are stale — they reflect the data as of the last run, so they cannot react to what a user is doing right now.
- Cannot handle new/unseen entities that arrive between runs (a brand-new user has no prediction yet — the cold-start problem).
- Wasted computation: predictions are produced for all entities even though only a fraction may ever be used.
- Requires storage for all predictions and a refresh strategy.

Suitable use cases: customer churn scoring run nightly, credit-risk scoring of an existing portfolio, product recommendation lists refreshed daily, demand/sales forecasting, lead scoring for a sales team, email campaign targeting, monthly customer segmentation, document classification of an archive.

Real-time inference (online inference)

The model is exposed as a live service (REST/gRPC endpoint) and generates a prediction for a single request on demand, typically within milliseconds.

Advantages:
- Immediate, fresh predictions based on the very latest input, including the current session context.
- Handles new users and new items instantly — no cold-start gap.
- Enables genuinely interactive products: personalised results, instant decisions, dynamic pricing.
- No wasted compute — a prediction is produced only when it is actually requested.
- No storage of unused predictions.

Disadvantages:
- Complex infrastructure: requires an always-on service, load balancing, autoscaling, health checks, versioning and blue-green/canary deployment.
- Higher and continuous cost because servers run 24×7 to absorb peak load.
- Strict latency budget limits model size and feature complexity; features must be fetched from a low-latency online store (Redis/DynamoDB).
- Higher risk of training-serving skew since features are computed by a different path than in training.
- Availability requirements: an outage directly breaks the user-facing product; needs retries, timeouts, fallbacks and caching.
- Monitoring is harder — ground-truth labels arrive much later than the predictions.

Suitable use cases: credit-card fraud detection at the moment of the transaction, loan approval decisions, ad click-through prediction and real-time bidding, search ranking and autocomplete, dynamic pricing (ride-hailing, airlines), chatbots and LLM responses, content feed personalisation, self-driving perception, real-time anomaly detection in infrastructure.

Other patterns worth mentioning
- Micro-batch / streaming inference (Kafka + Spark Structured Streaming, Flink): a middle ground with near-real-time latency of seconds.
- Hybrid approach (very common in practice): pre-compute expensive features and candidate sets in batch, then do a light real-time ranking step on top — this gives freshness with manageable latency and cost.
- Edge inference: the model runs on the device itself (mobile, IoT) for offline use and privacy.

Quick comparison
- Latency: batch = minutes to hours; real-time = milliseconds.
- Volume per call: batch = millions of rows; real-time = one or a few rows.
- Trigger: batch = a schedule; real-time = a user/system request.
- Freshness: batch = stale; real-time = current.
- Cost and complexity: batch = low; real-time = high.

## Question 4: What is model drift? Explain the difference between data drift, concept drift, and prediction drift, and describe why monitoring them is important.

**Answer:**

Model drift (model decay) is the degradation of a deployed model's predictive performance over time, caused by changes in the real world that make the relationships learned during training no longer valid. A model is trained on a static snapshot of the past, but it operates in a dynamic environment — customer behaviour, markets, competitors, regulations, product features and data pipelines all change. Every production model therefore decays; the only question is how fast.

### 1. Data drift (covariate shift / feature drift)
- Definition: The distribution of the input features changes, while the underlying relationship between inputs and the target stays the same. Formally, P(X) changes but P(Y|X) stays the same.
- Example: A churn model was trained when the average customer age was 35; a year later marketing attracted a younger base with an average age of 24. The features shifted, even though "young, low-engagement users churn more" is still true.
- Example: An e-commerce model sees a sudden jump in mobile traffic share, or a new geographic market is launched.
- Detection: Compare the training/reference distribution with the live distribution per feature using the Population Stability Index (PSI), Kolmogorov–Smirnov test (numeric), Chi-square test (categorical), Jensen–Shannon divergence, Wasserstein distance or KL divergence. A PSI above ~0.2 usually signals significant drift.
- Fix: Retrain on recent data, reweight samples, or update feature engineering. This can be detected without waiting for labels, which is its biggest practical advantage.

### 2. Concept drift
- Definition: The relationship between the inputs and the target changes — the meaning of the concept being predicted has shifted. Formally, P(Y|X) changes, whether or not P(X) changes. This is the most serious type of drift.
- Example: Before COVID-19, ordering groceries online was a signal of a "tech-savvy young" customer; after the pandemic, the same behaviour is universal, so the same feature values now imply a different outcome.
- Example: Fraudsters change their tactics, so patterns previously typical of legitimate transactions are now typical of fraud. Spam filters face the same adversarial concept drift.
- Types: Sudden/abrupt (a regulation change, a pandemic), gradual (slowly changing preferences), incremental (steady small shifts) and recurring/seasonal (patterns that return every festival season).
- Detection: Requires ground-truth labels, so it is monitored via a drop in accuracy/precision/recall/AUC over time, error-rate monitors such as DDM/EDDM/ADWIN/Page-Hinkley, and the fact that performance falls even when feature distributions look stable.
- Fix: Retrain on recent labelled data, use online/incremental learning, weight recent data more heavily, or redesign features to capture the new behaviour.

### 3. Prediction drift (model output drift)
- Definition: The distribution of the model's own outputs changes — the predicted classes, predicted probabilities or predicted values shift compared with the reference period. Formally, P(Ŷ) changes.
- Example: A fraud model that used to flag 2% of transactions suddenly flags 15%, or a loan model's average approval probability falls from 0.6 to 0.3.
- Example: A regression model's predicted demand values shift upward across the board.
- Detection: Monitor the distribution of predicted scores/classes over time using PSI or KS tests against the reference period, and monitor the positive-prediction rate.
- Why it is useful: It is the cheapest and fastest early-warning signal — available immediately, requires no labels, and is easy to compute. But it is a symptom, not a diagnosis: prediction drift can be caused by data drift, concept drift, a broken data pipeline, or even a legitimate change in reality, so it must be investigated alongside the other two.

Relationship between them
- Data drift changes the inputs → this usually causes prediction drift → which may or may not hurt accuracy.
- Concept drift changes the input–output relationship → this always hurts accuracy, and may occur with no visible data drift at all (the dangerous silent case).
- Prediction drift can also occur with no real drift — for example because of a data-quality bug that fills a feature with nulls.

Why monitoring drift is important
- Silent failure: an ML model does not crash when it becomes wrong. It keeps returning confident predictions, so without monitoring the business can lose money for months before anyone notices.
- Direct business and financial impact: undetected fraud, wrong credit decisions, poor recommendations, mispriced inventory.
- Early warning before labels arrive: ground truth (did the customer actually churn? was the loan repaid?) can take weeks or months. Data and prediction drift give a signal immediately.
- Triggering the right action: monitoring tells you *what* to fix — pipeline bug, retraining, feature redesign or full model redesign — instead of blindly retraining.
- Automated retraining: drift thresholds are the trigger condition in a continuous-training MLOps pipeline.
- Trust, fairness and compliance: drift can introduce bias against particular subgroups; regulators increasingly require documented ongoing model monitoring.
- Data-quality assurance: drift monitors also catch upstream pipeline breakages, schema changes and null explosions.
- Tools: Evidently AI, WhyLabs, Arize, Fiddler, NannyML, Deepchecks, AWS SageMaker Model Monitor, Prometheus + Grafana dashboards.

## Question 5: What is Docker, and why is it commonly used for deploying machine learning models? Also explain how scaling and rollback help maintain a reliable ML deployment.

**Answer:**

What is Docker?

Docker is an open-source containerisation platform that packages an application together with all of its dependencies — code, runtime, system libraries, environment variables and configuration — into a single portable unit called a container. A container is built from an image, which is defined by a Dockerfile and stored in a registry (Docker Hub, ECR, GCR).

Unlike a virtual machine, containers share the host operating-system kernel instead of running a full guest OS. They are therefore lightweight (megabytes rather than gigabytes), start in seconds, and allow many containers to run on one machine — while still being isolated from each other.

Why Docker is used for ML deployment
- Reproducibility — eliminates "it works on my machine". ML models are extremely sensitive to library versions; a model trained with scikit-learn 1.3 may fail to unpickle or may produce different results under 1.1. The image pins the exact Python version and every package version.
- Dependency isolation: ML stacks are heavy and conflict-prone (TensorFlow vs PyTorch, CUDA versions, numpy pins). Each model runs in its own container with its own dependency tree.
- Portability: the same image runs identically on a laptop, an on-prem server, AWS, GCP or Azure, which avoids cloud lock-in and makes local debugging identical to production.
- Consistency between environments: development, staging and production all run the same artifact, removing a whole class of deployment bugs.
- Fast, repeatable deployment: deploying becomes "pull image, run container" rather than a long manual environment setup.
- Ready for orchestration: containers are the unit that Kubernetes, ECS, Cloud Run, SageMaker and Vertex AI schedule, scale and heal.
- Resource control and isolation: CPU/memory limits per container, and GPU support via the NVIDIA container toolkit.
- Versioning of the whole environment: image tags (model-api:v1.2) version the code *and* the environment together, which is exactly what rollback needs.
- Microservices architecture: preprocessing, the model API and monitoring can each be a separate container, scaled and updated independently.
- MLflow integrates directly — mlflow models build-docker generates a serving image from a logged model.

Scaling

Scaling means adjusting capacity so the service keeps responding correctly as load changes.
- Horizontal scaling (scaling out): run more container replicas behind a load balancer. This is the standard approach for stateless model APIs, and is handled automatically by Kubernetes Horizontal Pod Autoscaler, ECS service autoscaling or Cloud Run.
- Vertical scaling (scaling up): give a container more CPU, RAM or a bigger GPU — useful for large models but limited by the size of a single machine.
- Autoscaling policies react to CPU/GPU utilisation, memory, request rate or queue length, and can scale to zero when idle to save cost.
- How scaling improves reliability: it absorbs traffic spikes (festival sales, viral events) without timeouts; it keeps latency (p95/p99) within SLA; it removes single points of failure because if one replica crashes the load balancer routes around it and the orchestrator restarts it; it enables rolling updates with zero downtime; and it controls cost by shrinking capacity during quiet periods.
- ML-specific considerations: model containers are memory-heavy (the model must be loaded per replica), cold starts are slow for large models (keep a warm pool), GPU replicas are expensive so batching requests is important, and readiness probes must wait until the model is actually loaded before traffic is sent.

Rollback

Rollback means reverting the service to a previously known-good version when a new deployment misbehaves.
- Why it is essential in ML: offline metrics do not guarantee online performance. A model with better test AUC can perform worse in production due to training-serving skew, a subtle feature bug, latency regressions or unexpected behaviour on a subgroup. Rollback is the safety net.
- Enabled by immutable versioning: every model is a tagged Docker image and a numbered version in the MLflow Model Registry, so the previous version still exists and can be restored exactly.
- Deployment strategies that make rollback safe:
- Blue-Green: run the old (blue) and new (green) environments side by side and switch traffic at the load balancer. Rollback is an instant traffic switch back.
- Canary: send a small percentage (1–5%) of traffic to the new model, watch metrics, then increase gradually. Damage is limited to a small share of users.
- Shadow / dark launch: send a copy of live traffic to the new model without using its output, purely to compare predictions and latency — zero user risk.
- A/B testing: split traffic to compare business metrics (conversion, revenue), not just ML metrics.
- Rolling update: replace replicas gradually; Kubernetes supports kubectl rollout undo to revert.
- Automated rollback: wire monitoring alerts (error rate, latency, prediction drift, accuracy) to trigger an automatic revert, so recovery takes seconds instead of hours.
- Together: scaling keeps the service available and fast under load, rollback keeps it correct and recoverable after a bad change. Both depend on Docker's immutable, versioned images, and together they are what turn a model into a dependable production system.

## Question 6: A binary classification model is deployed to production. After one month: Precision 91% → 78%, Recall 89% → 65%, Accuracy 94% → 91%. How would you investigate the degradation?

**Answer:**

Step 1: Read what the numbers are already telling us
- Recall has collapsed the most (−24 points), precision has fallen substantially (−13 points), but accuracy has barely moved (−3 points).
- This pattern is the classic signature of an imbalanced dataset. Accuracy is dominated by the majority (negative) class, so it hides the failure. Accuracy is the wrong headline metric here — F1, PR-AUC, ROC-AUC and the confusion matrix must be used instead.
- Falling recall means many more false negatives — the model is missing real positives. Falling precision means more false positives too. Both degrading at once means the model's discrimination ability has genuinely weakened, not that the decision threshold has merely shifted (a pure threshold shift usually trades one metric against the other).
- First quantify the business impact: what does each missed positive cost (undetected fraud, missed churn, missed disease)?

Step 2: Rule out measurement and label problems first
- Confirm the metrics are computed on a comparable population and time window — not a smaller sample, a different segment, or a period with unusual traffic.
- Check label availability and latency. If ground truth arrives late, recent positives may not be labelled yet, which artificially deflates recall. This is a very common false alarm.
- Check for a change in the labelling process — new annotators, a changed definition of the positive class, or a changed business rule.
- Verify the evaluation code itself did not change and that the threshold used in production is the one the metrics assume.

Step 3: Check data quality and the pipeline (the most frequent real cause)
- Compare the schema of incoming data against training: missing columns, renamed columns, reordered features, changed data types.
- Check null/missing rates per feature. A sudden spike of nulls in an important feature (an upstream API failing, a field deprecated) is a classic cause of this exact pattern.
- Check for unit or encoding changes — currency changed, seconds to milliseconds, a new category value that the encoder maps to "unknown".
- Check for duplicate or delayed records, timezone changes and clock skew.
- Verify the preprocessing/feature pipeline version deployed in production matches the one used at training (scaler fitted on old data, different imputation defaults) — i.e. look for training-serving skew.
- Confirm the correct model version is actually being served, and that no infrastructure change (library upgrade, container rebuild) altered behaviour.

Step 4: Test for data drift (no labels needed)
- For every feature, compare the production distribution against the training reference using PSI (flag > 0.1, act > 0.2), KS test for numeric features and Chi-square for categoricals; also compare means, medians and category frequencies.
- Rank features by drift magnitude weighted by feature importance — drift in a low-importance feature is usually harmless; drift in a top feature is the prime suspect.
- Check the class balance: has the base rate of positives changed? A change in prevalence alone mathematically changes precision even with an unchanged model.
- Check for new segments: a new country, channel, device type or product line that the model never saw in training.

Step 5: Test for concept drift
- If features look stable but performance has dropped, the input–output relationship has changed — concept drift.
- Retrain the *same* model architecture on recent labelled data. If the retrained model recovers performance on recent data, it confirms concept drift rather than a code bug.
- Look for external events in the period: a policy or pricing change, a competitor launch, a new fraud tactic, a seasonal effect, a marketing campaign that changed the user mix.
- Use drift detectors such as ADWIN, DDM or Page-Hinkley on the error stream to locate exactly when the change started.

Step 6: Slice the errors
- Break down precision and recall by segment — region, channel, device, customer tenure, product category, time of day. Often the degradation is concentrated in one new or fast-growing segment rather than spread evenly.
- Plot metrics over time (daily). A sudden cliff points to a release, a pipeline break or a sudden external event; a gradual slope points to genuine drift.
- Inspect the confusion matrix and a sample of the new false negatives manually — reading 50 missed cases usually reveals the pattern faster than any statistic.
- Examine the predicted-probability distribution: if scores are compressed toward the threshold, the model has lost confidence and separation.

Step 7: Diagnosis table and action
- Sudden drop + schema/null anomalies + feature distribution jump → data-quality / pipeline bug. Action: fix the pipeline, no retraining needed. Fastest and most common fix.
- Feature distributions shifted, relationship intact → data drift. Action: retrain on recent data, possibly reweight or add the new segment.
- Features stable, performance dropped, retraining on recent data recovers it → concept drift. Action: retrain, add features capturing the new behaviour, adopt more frequent/online retraining.
- Only precision moved and the positive rate changed → class-prevalence shift or threshold issue. Action: recalibrate probabilities (Platt/isotonic) and re-tune the threshold on recent data.
- Nothing changed externally but scores look odd → wrong model version or an environment/library change. Action: verify the artifact and roll back.

Immediate mitigation while investigating
- If recall is business-critical (fraud/medical), lower the decision threshold temporarily to recover recall at the cost of precision, and add human review for borderline cases.
- Consider rolling back to the previous model version if it still performs better.
- Set up alerts on PSI, null rates and daily F1 so the next degradation is caught in days, not a month — and schedule periodic retraining with an automated CT pipeline.

## Question 7: Why can a model that performs well during training and testing perform poorly in production?

**Answer:**

Offline evaluation happens on a clean, static, historical, balanced snapshot of data, under ideal conditions. Production is live, messy, changing and adversarial. The gap between the two has several distinct causes:

### 1. Data-related causes
- Data drift: the input distribution in production differs from training — new customer demographics, new markets, new devices, seasonality.
- Concept drift: the relationship between features and target changes over time (new fraud tactics, changed consumer behaviour, a policy change).
- Data-quality issues: production data has missing values, nulls from a failed upstream API, schema changes, wrong units, new unseen categories and duplicates that the curated training set never had.
- Unrepresentative or biased training data: the training sample did not reflect the real population — sampling bias, an over-cleaned dataset, or an artificially balanced set (SMOTE) while production is heavily imbalanced.
- Stale training data: the model was trained on data that is already months old by the time it is deployed.

### 2. Methodological causes (mistakes made during development)
- Data leakage — the single biggest cause of an inflated offline score. A feature contains information not available at prediction time (a field populated after the outcome, a target-derived column, an ID that encodes the label). Offline accuracy looks superb; production collapses.
- Preprocessing leakage: scaling, imputation or feature selection fitted on the *whole* dataset before splitting, so test data influenced the transformation.
- Improper splitting for temporal data: random train/test split on a time series lets the model "see the future". A chronological split is mandatory.
- Overfitting to the test set: hyperparameters tuned repeatedly against the same test set until the score is optimistically biased; no held-out final set.
- Overfitting the model itself: high variance, memorising training noise, insufficient regularisation or cross-validation.
- Duplicate records across train and test, which leaks identical rows into evaluation.
- Wrong evaluation metric: optimising accuracy on an imbalanced problem, so the model looks excellent while being useless on the minority class that actually matters.

### 3. Engineering and deployment causes
- Training-serving skew: features computed one way in the training notebook (Pandas) and another way in the serving service (SQL/Java) — different windows, rounding, null handling or time zones.
- Environment mismatch: different library versions between training and serving change numerical behaviour or break pickled objects.
- Latency constraints force the removal or simplification of expensive features in production that were present in training.
- Serving bugs: wrong feature order, wrong model version loaded, stale cache, incorrect threshold applied.
- Missing real-time feature availability: a feature available in the historical warehouse simply is not available at request time.

### 4. Real-world and business causes
- Feedback loops: the model's own predictions change user behaviour and therefore the future data (a recommender only ever gets feedback on what it recommended).
- Adversarial adaptation: fraudsters and spammers deliberately evolve to defeat the model.
- Edge cases and long-tail inputs that never appeared in training.
- Cold start: new users or items with no history.
- Business metric mismatch: the model optimises F1 while the business cares about revenue or cost-weighted errors; a technically good model can still be commercially poor.

How to prevent the gap
- Use a time-based split and a truly held-out final test set; audit every feature for leakage by asking "would this value exist at prediction time?"
- Build the preprocessing inside a Pipeline fitted only on training folds.
- Use a feature store to guarantee identical feature logic offline and online.
- Containerise with Docker and pin dependencies for environment parity.
- Run shadow deployment and canary releases to measure real production performance before full rollout.
- Evaluate with metrics aligned to business value, and validate on realistic class balance.
- Monitor drift, data quality and performance continuously, and retrain regularly.

## Question 8: What is a workflow orchestrator, and why is it required for production ML pipelines?

**Answer:**

A workflow orchestrator is a system that defines, schedules, executes and monitors multi-step workflows, managing the dependencies between the steps. Workflows are usually expressed as a DAG (Directed Acyclic Graph) where each node is a task and each edge is a dependency, so a task runs only after its prerequisites have succeeded.

Common tools: Apache Airflow, Prefect, Dagster, Kubeflow Pipelines, Metaflow, Luigi, Argo Workflows, AWS Step Functions, Azure Data Factory, Mage.

A typical ML pipeline DAG

In [ ]:
ingest_data → validate_data → preprocess → feature_engineering
            → train_model → evaluate_model → (gate: is it better?)
            → register_model → deploy → monitor


Why it is required for production ML
- Dependency management: ML steps are strictly ordered — you cannot train before features are built. The orchestrator enforces the order, runs independent branches in parallel, and stops downstream tasks when an upstream one fails.
- Automated scheduling and event triggers: pipelines run nightly, hourly, on a cron expression, or on an event such as new data landing in S3, a Kafka message, a Git commit or a drift alert — with no human running notebooks manually.
- Reliability: retries, timeouts and error handling. Transient failures (a database timeout, an API rate limit, a spot instance reclaimed) are retried automatically with backoff instead of silently killing the nightly run.
- Failure recovery and backfills: a failed pipeline can resume from the failed task rather than re-running hours of work, and historical periods can be backfilled systematically.
- Observability: a UI showing every run, its status, duration, logs and history, plus alerting to Slack/email/PagerDuty when something fails. Without this, failures are discovered by a business user noticing stale numbers.
- Reproducibility and versioning: the pipeline definition is code stored in Git, so the exact sequence of steps that produced a model is versioned and auditable — essential for compliance.
- Resource management and scalability: tasks are distributed across workers or Kubernetes pods, concurrency is limited to protect databases, and heavy training tasks can be routed to GPU nodes.
- Parameterisation and reuse: the same DAG runs for different dates, regions or models via parameters, avoiding copy-pasted scripts.
- Enables continuous training (CT): the orchestrator is what turns retraining from a manual chore into an automatic loop — drift detected → retrain → evaluate → promote only if better → deploy.
- Integration glue: it coordinates heterogeneous systems — Spark, dbt, a data warehouse, MLflow, Docker/Kubernetes, cloud storage and the serving layer — in one coherent workflow.
- Conditional logic and gates: "deploy only if the new model beats the current production model by at least 1% on the holdout set" can be encoded as a branch in the DAG.

What happens without one
- Manual notebook execution, forgotten steps and inconsistent runs.
- Fragile cron jobs and shell scripts with no dependency awareness, no retries and no visibility.
- Silent failures discovered days later, stale models in production, and no audit trail — which makes the system impossible to operate at scale or to pass a compliance review.

## Question 9: A model has an accuracy of 94% during testing but only 82% after three months in production. What would you investigate first, and what possible causes could explain the degradation?

**Answer:**

What I would investigate first (in priority order)
- Validate the measurement before assuming the model broke. Confirm the 82% is computed correctly — on the same population, the same class definition and the same threshold, with enough samples. Check whether ground-truth labels are complete; delayed labelling often makes recent performance look worse than it is. Rule out a bug in the monitoring/evaluation code itself.
- Plot accuracy day by day over the three months. The shape of the curve is the most informative single diagnostic: a sudden cliff on a specific date points to a deployment, a pipeline break or an external event, while a gradual downward slope points to genuine drift. Correlate any cliff with the release/change log.
- Check data quality and the input pipeline. Compare the production schema against training; inspect null and missing rates per feature, new unseen categorical values, unit or encoding changes, duplicates and timezone issues. Broken upstream data is the most common real cause and the fastest to fix.
- Confirm the right artifact is being served. Verify the model version, the preprocessing/feature pipeline version and the library versions in the container match what was validated — i.e. check for training-serving skew.
- Run data-drift tests (no labels required). Compute PSI, KS tests and Chi-square tests per feature against the training reference; prioritise the features with the highest importance. Also check whether the class balance/base rate has changed.
- Look beyond accuracy. On an imbalanced problem accuracy is misleading. Compute the confusion matrix, precision, recall, F1, ROC-AUC and PR-AUC, and inspect the predicted-probability distribution for compression or miscalibration.
- Slice the errors by segment — region, channel, device, customer tenure, product, new vs returning users. Degradation is often concentrated in one fast-growing or newly launched segment rather than spread evenly.
- Test for concept drift: retrain the same architecture on recent labelled data. If performance recovers, the relationship has changed rather than the code being broken.

Possible causes

A. Data drift (covariate shift) — the input distribution changed: a new marketing campaign brought a different customer profile, a new geography or channel was launched, seasonality (three months can cross a festival or quarter boundary), or the traffic mix shifted from desktop to mobile.

B. Concept drift — the input–target relationship changed: new competitor pricing, a policy or regulation change, evolving fraud tactics, or a shift in customer preferences. The model's learned rules are simply no longer true.

C. Data-quality and pipeline issues — an upstream API started returning nulls, a column was renamed or reordered, a unit changed (seconds to milliseconds, currency), a new category value is being encoded as "unknown", or an ETL job silently failed. These cause sharp, sudden drops.

D. Training-serving skew — features computed differently in production than in training; the scaler fitted on old statistics; different imputation defaults; a feature unavailable in real time being replaced with a default value.

E. Problems that existed at training time but were hidden — data leakage inflating the 94% (a feature not truly available at prediction time), a random split on temporal data, overfitting to the test set through repeated tuning, or an artificially balanced test set that does not match production prevalence. In these cases the model never really was 94%; production is showing its true performance.

F. Stale model and natural decay — the model was trained on data that is now six or more months old and has simply not been retrained.

G. Feedback loops and adversarial behaviour — the model's own decisions changed user behaviour, or bad actors adapted specifically to evade it.

H. Infrastructure/serving issues — timeouts causing fallback default predictions, a stale feature cache, partial outages, or the wrong model version deployed.

Actions after diagnosis
- Pipeline/data bug → fix the data source; no retraining required.
- Data drift → retrain on recent data, add the new segment, reweight recent samples.
- Concept drift → retrain frequently, add features that capture the new behaviour, consider online/incremental learning.
- Prevalence shift → recalibrate probabilities and re-tune the decision threshold.
- Leakage/overfitting → rebuild the model properly with a time-based split and a clean feature audit.
- Immediate mitigation → roll back to a previous better-performing version if one exists.
- Long term → put drift and data-quality monitoring with alert thresholds in place, and automate scheduled/triggered retraining so this is caught in days rather than three months.

## Question 10: Design a complete ML workflow for a customer churn prediction system.

The system must cover experiment tracking and best-model selection, deployment, monitoring, automated pipeline runs on new data, retraining on degradation, and rollback.

**Answer:**

Overall architecture

In [ ]:
Data Sources (CRM, transactions, support tickets, app logs)
        ↓
Ingestion + Validation  →  Feature Engineering  →  Feature Store (Feast)
        ↓
Training + Experiment Tracking (MLflow Tracking)
        ↓
Evaluation Gate  →  Model Registry (MLflow)  [None→Staging→Production]
        ↓
CI/CD (GitHub Actions)  →  Docker image  →  Serving (FastAPI on Kubernetes)
        ↓
Monitoring (Evidently + Prometheus/Grafana)
        ↓
Drift/Performance alert  →  Orchestrator (Airflow)  →  back to Training  [CT loop]


Stage 0: Data and feature layer (foundation)
- Ingest customer demographics, transaction history, usage/engagement logs, support-ticket history and subscription data into a warehouse (Snowflake/BigQuery) with Airflow scheduling the ingestion.
- Validate every batch with Great Expectations or Evidently — schema, null rates, ranges, category values. Fail the pipeline loudly if validation fails, because bad data is the most common cause of production failures.
- Engineer churn features: RFM (recency, frequency, monetary), tenure, usage trend over the last 30/60/90 days, support-ticket count and sentiment, payment failures, discount dependence, login gap.
- Store features in a feature store (Feast) with an offline store for training and an online store (Redis) for inference. This guarantees identical feature logic in both paths and provides point-in-time correct joins so no future information leaks into training.
- Version the data with DVC or Delta Lake so every training run points to an exact dataset snapshot.
- Critical for churn: use a time-based split, never a random one, and define the prediction window explicitly (e.g. "will this customer churn in the next 30 days, predicted as of today").

### 1. Track experiments and identify the best model
- Use MLflow Tracking. Each training run logs hyperparameters, metrics, artifacts (model, feature-importance plot, confusion matrix, PR curve), the Git commit, the data version and the environment. Enable autologging to capture everything with one line.
- Train several candidates — Logistic Regression as an interpretable baseline, Random Forest, XGBoost/LightGBM (usually the strongest on tabular churn data) — and tune with Optuna or Hyperopt, logging every trial as a nested run.
- Because churn is imbalanced (typically 5–20% churners), do not select on accuracy. Select on PR-AUC / F1 / Recall at a fixed precision, and use class weights, scale_pos_weight or careful resampling.
- Evaluate against business value, not just statistics: the cost of a missed churner (lost lifetime value) versus the cost of a wasted retention offer. Tune the decision threshold to maximise expected profit, and measure lift in the top decile, which is what the retention team actually acts on.
- Compare runs in the MLflow UI, then register the winner to the Model Registry with a description, the evaluation report and a link to its run.
- Add explainability with SHAP so the retention team understands *why* a customer is flagged — essential for adoption and for fairness checks across demographic groups.

### 2. Deploy the model for making predictions
- Hybrid serving is the right choice for churn. Run a nightly batch job that scores the entire customer base and writes churn scores to a database, since retention campaigns are planned in batches — this is cheap and allows heavy features. Additionally expose a real-time REST endpoint for on-demand scoring, for example when an agent opens a customer record or a cancellation flow is triggered.
- Package the model with Docker — pinned dependencies, the model loaded from models:/churn_model/Production so the container always serves the currently promoted version. Serve with FastAPI (+ Gunicorn/Uvicorn) or MLflow models serve / BentoML / Seldon.
- Deploy onto Kubernetes (or SageMaker/Vertex AI/Cloud Run) with a load balancer, horizontal pod autoscaling, readiness probes that wait for the model to load, and resource limits.
- Automate with CI/CD (GitHub Actions): on a merge to main — run unit tests, data-validation tests and model tests — build and push the Docker image → deploy to staging → run integration and load tests → promote to production on approval.
- Roll out safely using shadow deployment first (mirror traffic, compare predictions, no user impact), then a canary at 5–10% of traffic, then blue-green for the full switch. Always keep the previous version running until the new one is proven.
- Log every prediction (input features, score, model version, timestamp) to a prediction store — this is the raw material for monitoring and for the next training set.

3. Monitor the model and detect data or performance problems
- Data quality monitoring: schema checks, null and missing rates, unexpected categories, volume anomalies, freshness/latency of upstream tables.
- Data drift monitoring: per-feature PSI (alert > 0.2), KS tests for numeric and Chi-square for categorical features, weighted by feature importance. Use Evidently AI, NannyML, WhyLabs or SageMaker Model Monitor.
- Prediction drift monitoring: track the distribution of churn scores and the predicted churn rate. A jump from 8% to 20% flagged churners is an immediate early warning — it needs no labels.
- Performance monitoring (delayed labels): churn labels arrive only after the prediction window closes (e.g. 30–90 days). Maintain a delayed-label join so that when ground truth arrives, precision, recall, F1, PR-AUC and top-decile lift are computed on that cohort. Track them as a rolling time series.
- Segment monitoring: compute metrics per plan type, tenure band, region and acquisition channel, so degradation concentrated in one segment is not hidden by the overall average.
- Operational monitoring: latency p95/p99, error rate, throughput, CPU/memory, container restarts — with Prometheus + Grafana.
- Business monitoring: retention campaign conversion rate, revenue saved, and offer cost — the metrics the business actually cares about.
- Alerting: thresholds wired to Slack/PagerDuty, e.g. PSI > 0.2 on a top-5 feature, predicted churn rate outside ±30% of baseline, F1 below 0.70, null rate > 5%, or p99 latency > 200 ms.

4. Automatically run the ML pipeline when new data becomes available
- Use a workflow orchestrator — Apache Airflow (or Prefect, Dagster, Kubeflow Pipelines) to define the whole pipeline as a DAG in version-controlled code.
- DAG: ingest → validate → build features → materialise to feature store → train → evaluate → gate → register → deploy → score → monitor.
- Triggers: a scheduled run (nightly scoring, weekly/monthly retraining); an event trigger such as a file landing in S3 (S3 event → Lambda → Airflow API) or a warehouse partition completing, detected by an Airflow sensor; a Kafka message; a Git commit via CI/CD; or a drift alert from the monitoring service calling the Airflow REST API.
- Configure retries with exponential backoff, task timeouts, SLA alerts and failure notifications so transient errors do not silently break the nightly run.
- Support backfills so historical periods can be recomputed, and parameterise the DAG by date and segment for reuse.

5. Retrain the model when performance decreases
- Operate a Continuous Training (CT) loop with three trigger types: scheduled (a monthly baseline retrain), performance-based (F1 or top-decile lift falls below a threshold on a labelled cohort), and drift-based (PSI above threshold on important features, or significant prediction drift).
- Retraining pipeline: pull the latest labelled window (for example a rolling 12–18 months, optionally weighting recent data more heavily) → validate the data → rebuild features → retrain and re-tune → log everything to MLflow as a new run.
- Champion–challenger evaluation gate: the retrained challenger is compared against the current production champion on the same recent holdout set. Promote to Staging only if it improves the primary metric by a meaningful margin (e.g. ≥ 1–2% F1) and does not regress on any key segment or fairness check. Otherwise keep the champion and raise an alert for human investigation — automatic promotion of a worse model must be impossible.
- Promote through the MLflow Model Registry stages (None → Staging → Production → Archived), with the Staging → Production transition requiring either automated test success or human approval, depending on risk appetite.
- Deploy the promoted model via canary/shadow first, watch the online metrics, then complete the rollout.
- Guard against runaway loops: rate-limit retraining frequency, and always require the evaluation gate to pass.

6. Roll back to the previous model if the new model performs poorly
- Rollback is possible only because everything is versioned and immutable: each model version in the MLflow Registry, each Docker image tag, each data snapshot in DVC and each code commit in Git.
- Manual/automated rollback mechanism: transition the previous version back to Production in the Registry and redeploy, or with blue-green simply switch traffic back to the still-running old environment — which is effectively instant. On Kubernetes, kubectl rollout undo reverts the deployment.
- Automated rollback triggers: error rate or latency exceeding the SLA, predicted churn rate outside expected bounds, a sharp fall in early business metrics (campaign conversion), or health-check failures. Wire the monitoring alerts directly to the rollback job so recovery takes seconds.
- Canary limits the blast radius: because only 5–10% of traffic sees the new model first, a rollback affects only that slice, and the decision to roll back can be made on statistically monitored canary metrics.
- Keep the previous version warm (running but not receiving traffic) during the canary window so failover is immediate.
- After any rollback, run a post-mortem: reproduce the issue from the logged run, data version and prediction logs; fix the root cause; add a regression test to CI so the same failure cannot recur.

Summary of the tool stack
- Orchestration: Apache Airflow
- Experiment tracking & registry: MLflow
- Data/feature management: DVC or Delta Lake, Feast feature store, Great Expectations for validation
- Training: scikit-learn / XGBoost / LightGBM with Optuna, SHAP for explainability
- Packaging & deployment: Docker, FastAPI, Kubernetes, GitHub Actions for CI/CD
- Monitoring: Evidently AI (drift & data quality), Prometheus + Grafana (operational), custom business dashboards
- Storage: Snowflake/BigQuery or S3 for offline, Redis for online features, a prediction log table for monitoring

The result is a closed, self-healing loop: new data → automated pipeline → tracked experiments → gated promotion → safe canary deployment → continuous monitoring → drift or performance alert → automated retraining → rollback if anything regresses. Human judgement stays in the loop at the promotion gate, but no routine step depends on someone remembering to run a notebook.